# Alerta de riesgo financiero empresarial

Este notebook ejecuta el pipeline completo en Google Colab y muestra los resultados principales.

Cambia `REPO` por la dirección de tu repositorio. Para probar sin descargar datos, deja `MODO = "demo"`.

In [ ]:
REPO = "https://github.com/<tu-usuario>/<nombre-del-repositorio>.git"
MODO = "demo"   # "demo" o "reales"

!git clone -q $REPO proyecto
%cd proyecto
!pip install -q -r requirements.txt

## 1. Ejecutar el pipeline

Con datos reales la primera descarga puede tardar varios minutos. Si tienes un token de datos.gov.co, defínelo antes en `os.environ["SOCRATA_APP_TOKEN"]`.

In [ ]:
import subprocess
args = ["python", "-m", "src.pipeline"] + (["--demo"] if MODO == "demo" else [])
r = subprocess.run(args, capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-3000:])

## 2. Métricas en el año de prueba

In [ ]:
import json, pandas as pd
resumen = json.load(open("docs/data/resumen.json", encoding="utf-8"))
pd.DataFrame(resumen["modelo"]["metricas"]).set_index("modelo")

In [ ]:
resumen["modelo"]["altman_zonas"]

## 3. Curvas ROC

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 5))
nombres = {"altman": "Z'' de Altman", "logit": "Logística (razones de Altman)", "xgboost": "XGBoost"}
for clave, puntos in resumen["modelo"]["roc"].items():
    x, y = zip(*puntos)
    ax.plot(x, y, label=nombres[clave])
ax.plot([0, 1], [0, 1], "--", color="grey")
ax.set_xlabel("Empresas sanas marcadas por error")
ax.set_ylabel("Deterioros detectados")
ax.legend()
plt.show()

## 4. Importancia SHAP

In [ ]:
pd.DataFrame(resumen["modelo"]["importancia_familias"]).plot.barh(x="familia", y="valor", legend=False, figsize=(6, 3))
plt.gca().invert_yaxis()
plt.show()
pd.DataFrame(resumen["modelo"]["importancia_variables"]).head(12)

## 5. Semáforo por sector

In [ ]:
pd.DataFrame(resumen["por_sector"]).set_index("nombre")

## 6. Tabla empresa-año para explorar

`y` vale 1 si la empresa pasó de sana a dificultad al año siguiente.

In [ ]:
ruta = "data/demo/tabla_empresas.parquet" if MODO == "demo" else "data/tabla_empresas.parquet"
tabla = pd.read_parquet(ruta)
tabla.groupby("anio").agg(empresas=("nit", "size"), en_dificultad=("dificultad", "mean"), tasa_deterioro=("y", "mean"))

## 7. Revisar los rubros que se usaron

Si algún indicador se ve extraño, confirma aquí qué rubro de la taxonomía NIIF quedó asignado a cada variable.

In [ ]:
pd.read_csv(ruta.replace("tabla_empresas.parquet", "reportes/conceptos_mapeados.csv"))